# ESD Inference for FLUX.2 Klein

This notebook compares the original `Flux2KleinPipeline` against an ESD-edited checkpoint.

The defaults below target `black-forest-labs/FLUX.2-klein-base-4B`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from diffusers import Flux2KleinPipeline

repo_root = Path.cwd().resolve()
if not (repo_root / "utils").exists():
    candidate = repo_root.parent
    if (candidate / "utils").exists():
        repo_root = candidate

if not (repo_root / "utils").exists():
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from utils.esd_checkpoint import apply_esd_checkpoint

torch.set_grad_enabled(False)


def infer_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def infer_dtype(device: str) -> torch.dtype:
    if device == "cuda":
        return torch.bfloat16
    if device == "mps":
        return torch.float16
    return torch.float32


def make_generator(device: str, seed: int) -> torch.Generator:
    target_device = torch.device(device)
    if target_device.type == "cuda":
        return torch.Generator(device=target_device).manual_seed(seed)
    return torch.Generator().manual_seed(seed)


In [ ]:
basemodel_id = "black-forest-labs/FLUX.2-klein-base-4B"
esd_path = repo_root / "esd-models" / "flux2-klein" / "your-checkpoint.safetensors"

prompt = "image of a church"
num_inference_steps = 50
guidance_scale = 4.0
max_sequence_length = 512
seed = 0

device = infer_device()
torch_dtype = infer_dtype(device)

print(f"repo_root={repo_root}")
print(f"device={device}, torch_dtype={torch_dtype}")
print(f"base_model={basemodel_id}")
print(f"esd_path={esd_path}")

In [ ]:
base_pipe = Flux2KleinPipeline.from_pretrained(
    basemodel_id,
    torch_dtype=torch_dtype,
    use_safetensors=True,
).to(device)
base_pipe.set_progress_bar_config(disable=True)

with torch.inference_mode():
    original_image = base_pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=make_generator(device, seed),
        max_sequence_length=max_sequence_length,
    ).images[0]

original_image

In [ ]:
del base_pipe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not esd_path.exists():
    raise FileNotFoundError(f"Set `esd_path` to a valid checkpoint first: {esd_path}")

esd_pipe = Flux2KleinPipeline.from_pretrained(
    basemodel_id,
    torch_dtype=torch_dtype,
    use_safetensors=True,
).to(device)
esd_pipe.set_progress_bar_config(disable=True)

metadata, resolved_component, load_result = apply_esd_checkpoint(
    esd_pipe,
    str(esd_path),
    device="cpu",
)

print(f"Loaded ESD weights into pipe.{resolved_component}")
print("Metadata:", metadata)
print(
    f"missing_keys={len(load_result.missing_keys)}, unexpected_keys={len(load_result.unexpected_keys)}"
)

with torch.inference_mode():
    esd_image = esd_pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=make_generator(device, seed),
        max_sequence_length=max_sequence_length,
    ).images[0]

esd_image

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(original_image)
axes[0].set_title("Original")
axes[1].imshow(esd_image)
axes[1].set_title("ESD edited")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
output_dir = repo_root / "esd-images" / "flux2-klein"
output_dir.mkdir(parents=True, exist_ok=True)

original_path = output_dir / "original.png"
esd_path_out = output_dir / "edited.png"

original_image.save(original_path)
esd_image.save(esd_path_out)

print(f"Saved original image to {original_path}")
print(f"Saved edited image to {esd_path_out}")